# 01 · Exploratory data analysis

Explore the collected snapshots with DuckDB. Run `git pull --rebase` first so you have the latest data.

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os

os.chdir(ROOT)
import pandas as pd

pd.set_option("display.width", 160)

In [ ]:
from velib import analytics

con = analytics.connect()
con.sql(
    "SELECT count(*) AS rows, count(DISTINCT slot_ts) AS snapshots, min(local_ts) AS first, max(local_ts) AS last FROM stg_snapshots"
).df()

## Network rhythm: share of stations empty / full by hour

In [ ]:
marts = analytics.build_marts(con)
prof = marts["mart_network_profile"]
prof.pivot(index="local_hour", columns="day_type", values="share_stations_empty").plot(
    title="Share of stations empty", figsize=(9, 4)
);

In [ ]:
prof.pivot(index="local_hour", columns="day_type", values="share_stations_full").plot(
    title="Share of stations full", figsize=(9, 4)
);

## Worst stations and areas

In [ ]:
marts["mart_station_reliability"].sort_values("problem_rank").head(20)[
    ["station_code", "name", "area", "share_time_empty", "share_time_full", "empty_hours_per_day"]
]

In [ ]:
marts["mart_area_summary"].head(15)

## Weather

In [ ]:
marts["mart_weather_effect"]

## Data quality: do we have every 15-minute slot?

In [ ]:
marts["mart_data_quality"]